# Visión por Computador — Práctica 3.1 / 3.2
## Parte 1 — Análisis comparativo de robustez: **Momentos de Hu vs Zernike vs Shape Signature**

**Universidad Politécnica Salesiana** · Docente: Ing. Vladimir Robles Bykbaev
**Estudiante:** Lata Zambrano Michael Israel — **Grupo G3**
**Dataset:** UPS-Writing-Skills (clases: `circle`, `square`, `triangle`)
**Parámetros personalizados (pista):** umbral de binarización **T = 130**, KNN **K = 7**

> **Cuaderno autónomo para Kaggle.** Todo el código (segmentación, descriptores,
> aumento de datos y clasificación) está **embebido en este notebook**: no depende
> de ningún paquete local. Solo requiere añadir el dataset
> [`ups-writing-skills`](https://www.kaggle.com/datasets/adolfogavilanes/ups-writing-skills)
> con **Add Data**. Corre completo con *Run All*.


### Fundamento teórico (Unidad 3.1 y 3.2)

**Momentos de Hu (Unidad 3.1).** A partir de los momentos RAW
$M_{ij}=\sum_x\sum_y x^i y^j I(x,y)$ se obtienen el centroide, los momentos
centrales $\mu_{pq}$ (invariantes a traslación), los normalizados $\eta_{ij}$
(invariantes a escala) y los **7 invariantes** $I_1..I_7$ (invariantes a rotación).
Se aplica la **transformación logarítmica** $h'=-\,\mathrm{sign}(h)\log_{10}|h|$
para igualar los órdenes de magnitud.

**Momentos de Zernike (Unidad 3.1).** Proyección de la imagen sobre los polinomios
**ortogonales** de Zernike dentro del círculo unitario,
$A_{nm}=\frac{n+1}{\pi}\sum I(x,y)\,\overline{V_{nm}}$, con
$V_{nm}=R_{nm}(\rho)e^{jm\theta}$. Se usa **n = 8**. Las **magnitudes** $|A_{nm}|$
son invariantes a la rotación; la ortogonalidad reduce la redundancia y mejora la
robustez al ruido.

**Shape Signature (Unidad 3.2).** Firma de **distancia al centroide** del contorno,
$r(k)$, transformada con **FFT**; se usan **13 componentes** (DC para normalizar +
12 como descriptor). La magnitud de la FFT da invarianza a rotación y punto inicial,
y la división por el DC da invarianza a escala.


## 0. Configuración e imports (embebido)

In [ ]:
import os
from pathlib import Path
from functools import lru_cache
from math import factorial, pi
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import matplotlib.pyplot as plt
from types import SimpleNamespace

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA

# Parámetros del proyecto (mismos que en src/utils/config.py).
config = SimpleNamespace(
    RANDOM_SEED=42,
    T_THRESHOLD=130,            # umbral fijo de binarización (pista G3)
    KNN_K=7,                    # vecinos KNN (pista G3)
    NORMALIZE_SIZE=256,
    MEDIAN_KSIZE=3,             # filtro de mediana contra Salt & Pepper
    UPS_CLASSES=["circle", "square", "triangle"],
    GAUSSIAN_LEVELS={"bajo": 0.02, "medio": 0.06, "alto": 0.12},
    SALT_PEPPER_LEVELS={"bajo": 0.02, "medio": 0.06, "alto": 0.12},
    AUG_VARIANTS_PER_IMAGE=6,
    ZERNIKE_DEGREE=8,           # grado del polinomio n=8
    SHAPE_SIG_N_HARMONICS=13,   # DC + 12 descriptor
)
print("T =", config.T_THRESHOLD, "| K =", config.KNN_K, "| Zernike n =", config.ZERNIKE_DEGREE)


## 1. Descriptores (embebido): Hu, Zernike (desde cero) y Shape Signature

In [ ]:
import cv2
import numpy as np


def hu_moments(mask: np.ndarray) -> np.ndarray:
    """
    Calcula los 7 momentos de Hu de una máscara binaria y devuelve su versión
    log-transformada (vector de 7 floats).

    Entrada:
        mask : máscara binaria uint8 (figura rellena, 255 = objeto).
    """
    # cv2.moments calcula M_ij, mu_ij y nu_ij (η) del documento de clase.
    m = cv2.moments(mask, binaryImage=True)
    hu = cv2.HuMoments(m).flatten()   # 7 invariantes I1..I7

    # Transformación logarítmica (igualar órdenes de magnitud).
    eps = 1e-30
    hu_log = -np.sign(hu) * np.log10(np.abs(hu) + eps)
    return hu_log.astype(np.float64)


# Nombre legible de cada componente, útil para tablas/figuras del reporte.
HU_FEATURE_NAMES = [f"Hu{i}" for i in range(1, 8)]

In [ ]:
from functools import lru_cache
from math import factorial, pi
from typing import List, Tuple

import numpy as np



def _valid_nm(max_degree: int) -> List[Tuple[int, int]]:
    """
    Lista de pares (n, m) válidos con m >= 0 (se usa m>=0 porque
    |A_{n,-m}| = |A_{n,m}|, evitando redundancia), |m|<=n y (n-m) par.
    """
    pairs = []
    for n in range(max_degree + 1):
        for m in range(0, n + 1):
            if (n - m) % 2 == 0:
                pairs.append((n, m))
    return pairs


def _radial_poly(n: int, m: int, rho: np.ndarray) -> np.ndarray:
    """Evalúa R_nm(ρ) (vectorizado sobre el arreglo rho)."""
    m = abs(m)
    R = np.zeros_like(rho)
    for s in range((n - m) // 2 + 1):
        c = ((-1) ** s) * factorial(n - s)
        c /= (factorial(s)
              * factorial((n + m) // 2 - s)
              * factorial((n - m) // 2 - s))
        R += c * rho ** (n - 2 * s)
    return R


@lru_cache(maxsize=4)
def _build_basis(size: int, max_degree: int):
    """
    Precalcula (y cachea) la base de Zernike para imágenes de 'size x size'.

    Se asume la figura CENTRADA en el lienzo (el preprocesamiento ya la centra)
    y un círculo unitario inscrito de radio = size/2.

    Devuelve:
        pairs : lista de (n,m)
        basis : arreglo complejo (len(pairs), K) con conj(V_nm) SÓLO en los
                píxeles dentro del círculo unitario (máscara aplanada).
        mask_disk : índices booleanos de los píxeles dentro del disco.
        norm  : factores (n+1)/π por cada par.
    """
    # Coordenadas normalizadas: centro del lienzo -> (0,0); borde -> radio 1.
    radius = size / 2.0
    ys, xs = np.mgrid[0:size, 0:size].astype(np.float64)
    x = (xs - radius) / radius
    y = (ys - radius) / radius

    rho = np.sqrt(x * x + y * y)
    theta = np.arctan2(y, x)
    disk = rho <= 1.0   # sólo el interior del círculo unitario

    rho_d = rho[disk]
    theta_d = theta[disk]

    pairs = _valid_nm(max_degree)
    basis = np.empty((len(pairs), rho_d.size), dtype=np.complex128)
    norm = np.empty(len(pairs), dtype=np.float64)

    for i, (n, m) in enumerate(pairs):
        R = _radial_poly(n, m, rho_d)
        V = R * np.exp(1j * m * theta_d)     # V_nm
        basis[i] = np.conj(V)                # se proyecta con conj(V)
        norm[i] = (n + 1) / pi

    return pairs, basis, disk, norm


def zernike_moments(mask: np.ndarray,
                    max_degree: int = config.ZERNIKE_DEGREE) -> np.ndarray:
    """
    Calcula el descriptor de Zernike (magnitudes |A_nm|) de una máscara binaria.

    Entrada:
        mask : máscara binaria cuadrada (size x size) uint8 {0,255}, figura
               centrada (salida de preprocessing.filled_shape_mask).
    Salida:
        vector de magnitudes |A_nm| (float64), longitud = nº de pares (n,m).
    """
    size = mask.shape[0]
    if mask.shape[0] != mask.shape[1]:
        raise ValueError("La máscara debe ser cuadrada para Zernike.")

    pairs, basis, disk, norm = _build_basis(size, max_degree)

    # Imagen normalizada a {0,1} dentro del disco.
    img = (mask > 0).astype(np.float64)[disk]

    # Proyección: A_nm = (n+1)/π · Σ I · conj(V_nm)
    A = basis @ img            # producto vectorizado (suma sobre píxeles)
    A = A * norm
    return np.abs(A)


def zernike_feature_names(max_degree: int = config.ZERNIKE_DEGREE) -> List[str]:
    """Nombres legibles 'Z_n_m' para las tablas del reporte."""
    return [f"Z{n}_{m}" for (n, m) in _valid_nm(max_degree)]

In [ ]:
import cv2
import numpy as np



def resample_contour(contour: np.ndarray, n_points: int) -> np.ndarray:
    """
    Remuestrea un contorno cerrado (N,2) a 'n_points' puntos equiespaciados a lo
    largo de su perímetro (interpolación lineal por longitud de arco).
    """
    pts = contour.astype(np.float64)
    # Distancias acumuladas a lo largo del contorno (cerrado).
    closed = np.vstack([pts, pts[0]])
    deltas = np.diff(closed, axis=0)
    seg_len = np.sqrt((deltas ** 2).sum(axis=1))
    cum = np.concatenate([[0], np.cumsum(seg_len)])
    total = cum[-1]
    if total <= 0:
        # Contorno degenerado: replicar el primer punto.
        return np.repeat(pts[:1], n_points, axis=0)

    target = np.linspace(0, total, n_points, endpoint=False)
    x = np.interp(target, cum, closed[:, 0])
    y = np.interp(target, cum, closed[:, 1])
    return np.stack([x, y], axis=1)


def centroid_distance_signature(contour: np.ndarray, n_points: int) -> np.ndarray:
    """Calcula r(k) = distancia de cada punto remuestreado al centroide."""
    pts = resample_contour(contour, n_points)
    centroid = pts.mean(axis=0)
    diff = pts - centroid
    return np.sqrt((diff ** 2).sum(axis=1))


def shape_signature_descriptor(contour: np.ndarray,
                               n_harmonics: int = config.SHAPE_SIG_N_HARMONICS,
                               n_points: int = 256) -> np.ndarray:
    """
    Descriptor Shape Signature (12 valores) a partir de un contorno (N,2).

    n_harmonics = 13  ->  F(0) normaliza + F(1..12) descriptor.
    """
    r = centroid_distance_signature(contour, n_points)

    # FFT de la firma 1-D y su magnitud.
    F = np.fft.fft(r)
    mag = np.abs(F)

    dc = mag[0] if mag[0] > 1e-12 else 1.0     # componente DC (|F(0)|)
    # Tomamos F(1)..F(n_harmonics-1) y normalizamos por DC.
    descriptor = mag[1:n_harmonics] / dc
    return descriptor.astype(np.float64)


def shape_signature_feature_names(n_harmonics: int = config.SHAPE_SIG_N_HARMONICS):
    return [f"SS_F{k}" for k in range(1, n_harmonics)]

## 2. Segmentación (embebido): silueta rellena (Hu/Zernike) y contorno (Shape Signature)

In [ ]:
from typing import Optional

import cv2
import numpy as np



def to_binary(gray: np.ndarray,
              threshold: int = config.T_THRESHOLD,
              median_ksize: int = config.MEDIAN_KSIZE) -> np.ndarray:
    """
    Segmenta la tinta con umbral fijo. Devuelve máscara binaria uint8 {0,255}
    donde 255 = trazo (tinta oscura, gris < T).
    """
    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    if median_ksize and median_ksize >= 3:
        gray = cv2.medianBlur(gray, median_ksize)          # atenúa Salt & Pepper
    _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY_INV)
    return binary


def _significant_ink(binary: np.ndarray, min_area_frac: float = 0.05) -> Optional[np.ndarray]:
    """
    Conserva las componentes conexas cuya área sea >= min_area_frac de la mayor
    componente (une los fragmentos del trazo y descarta motas de ruido).
    """
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n <= 1:
        return None
    areas = stats[1:, cv2.CC_STAT_AREA]
    a_max = int(areas.max())
    keep = np.zeros_like(binary)
    for j in range(1, n):
        if stats[j, cv2.CC_STAT_AREA] >= min_area_frac * a_max:
            keep[labels == j] = 255
    return keep


def silhouette(gray: np.ndarray, threshold: int = config.T_THRESHOLD) -> Optional[np.ndarray]:
    """
    Silueta SÓLIDA (envolvente convexa del trazo) sin normalizar tamaño.
    Devuelve máscara uint8 {0,255} o None si no se detecta figura.
    """
    binary = to_binary(gray, threshold)
    # Pequeño cierre para unir trazos casi contiguos antes de filtrar.
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, ker, iterations=1)

    ink = _significant_ink(binary)
    if ink is None:
        return None
    pts = cv2.findNonZero(ink)
    if pts is None or len(pts) < 3:
        return None

    hull = cv2.convexHull(pts)
    out = np.zeros_like(binary)
    cv2.fillConvexPoly(out, hull, 255)
    return out


def largest_contour(binary: np.ndarray) -> Optional[np.ndarray]:
    """Contorno externo de mayor área (N,2) o None. RETR_EXTERNAL."""
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    return max(contours, key=cv2.contourArea).reshape(-1, 2)


def filled_shape_mask(gray: np.ndarray,
                      size: int = config.NORMALIZE_SIZE,
                      threshold: int = config.T_THRESHOLD) -> Optional[np.ndarray]:
    """
    Silueta rellena, centrada en el centroide y escalada al disco unitario.
    Devuelve máscara cuadrada (size x size) uint8 {0,255}, o None.
    """
    sil = silhouette(gray, threshold)
    if sil is None:
        return None
    cnt = largest_contour(sil)
    if cnt is None or cv2.contourArea(cnt) < 1.0:
        return None

    m = cv2.moments(sil, binaryImage=True)
    if m["m00"] <= 0:
        return None
    cx, cy = m["m10"] / m["m00"], m["m01"] / m["m00"]

    d = cnt.astype(np.float64) - np.array([cx, cy])
    max_r = float(np.sqrt((d ** 2).sum(axis=1)).max())
    if max_r <= 0:
        return None

    target_r = (size / 2.0) * 0.95
    scale = target_r / max_r
    M = np.array([[scale, 0.0, size / 2.0 - scale * cx],
                  [0.0, scale, size / 2.0 - scale * cy]], dtype=np.float64)
    return cv2.warpAffine(sil, M, (size, size), flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def preprocessed_contour(gray: np.ndarray,
                         threshold: int = config.T_THRESHOLD) -> Optional[np.ndarray]:
    """
    Contorno externo (N,2) de la silueta sólida, para Shape Signature. None si no
    hay figura. Usar la silueta (en vez del trazo crudo) da un contorno limpio y
    cerrado.
    """
    sil = silhouette(gray, threshold)
    if sil is None:
        return None
    return largest_contour(sil)

## 3. Aumento de datos (embebido): inyección de ruido y rotaciones 0–360°

In [ ]:
import numpy as np



def add_gaussian_noise(gray: np.ndarray, sigma_rel: float,
                       rng: np.random.Generator) -> np.ndarray:
    """
    Ruido Gaussiano aditivo. sigma_rel es la desviación estándar relativa a 255.
    g'(x,y) = g(x,y) + N(0, sigma)   con recorte a [0,255].
    """
    sigma = sigma_rel * 255.0
    noise = rng.normal(0.0, sigma, size=gray.shape)
    out = gray.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)


def add_salt_pepper(gray: np.ndarray, amount: float,
                    rng: np.random.Generator) -> np.ndarray:
    """
    Ruido impulsivo Salt & Pepper. 'amount' es la fracción total de píxeles
    alterados (mitad sal=255, mitad pimienta=0).
    """
    out = gray.copy()
    n = gray.size
    n_affected = int(amount * n)
    if n_affected == 0:
        return out

    flat = out.reshape(-1)
    idx = rng.choice(n, size=n_affected, replace=False)
    half = n_affected // 2
    flat[idx[:half]] = 255   # sal
    flat[idx[half:]] = 0     # pimienta
    return flat.reshape(gray.shape)


def apply_noise(gray: np.ndarray, noise_type: str, level: str,
                rng: np.random.Generator) -> np.ndarray:
    """
    Despachador genérico.
        noise_type : 'gaussian' | 'salt_pepper'
        level      : 'bajo' | 'medio' | 'alto'
    """
    if noise_type == "gaussian":
        return add_gaussian_noise(gray, config.GAUSSIAN_LEVELS[level], rng)
    if noise_type == "salt_pepper":
        return add_salt_pepper(gray, config.SALT_PEPPER_LEVELS[level], rng)
    raise ValueError(f"Tipo de ruido desconocido: {noise_type}")

In [ ]:
import cv2
import numpy as np


def rotate(gray: np.ndarray, angle_deg: float,
           border_value: int = 255) -> np.ndarray:
    """
    Rota la imagen 'angle_deg' grados alrededor de su centro, expandiendo el
    lienzo para no recortar la figura. El fondo se rellena con 255 (papel claro)
    para mantener la convención tinta-oscura / fondo-claro.
    """
    h, w = gray.shape[:2]
    cx, cy = w / 2.0, h / 2.0
    M = cv2.getRotationMatrix2D((cx, cy), angle_deg, 1.0)

    # Calcular el nuevo tamaño del lienzo tras la rotación.
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)
    M[0, 2] += (new_w - w) / 2.0
    M[1, 2] += (new_h - h) / 2.0

    return cv2.warpAffine(gray, M, (new_w, new_h),
                          flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT,
                          borderValue=border_value)


def random_rotation(gray: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Aplica una rotación aleatoria uniforme en [0, 360)."""
    angle = float(rng.uniform(0.0, 360.0))
    return rotate(gray, angle)

## 4. Fachada de extracción, pipeline de aumento y clasificador (embebido)

In [ ]:
from typing import Dict, Optional

import numpy as np



def extract_all(gray: np.ndarray) -> Optional[Dict[str, np.ndarray]]:
    """
    Extrae los descriptores Hu, Zernike y Shape Signature de una imagen en gris.
    Devuelve None si no se detecta figura (contorno/máscara vacíos).
    """
    mask = filled_shape_mask(gray)        # región rellena (Hu, Zernike)
    contour = preprocessed_contour(gray)  # contorno externo (Shape Signature)
    if mask is None or contour is None or len(contour) < 5:
        return None

    return {
        "hu": hu_moments(mask),
        "zernike": zernike_moments(mask),
        "shape_signature": shape_signature_descriptor(contour),
    }


DESCRIPTOR_NAMES = ["hu", "zernike", "shape_signature"]

In [ ]:
from typing import Dict, List, Tuple

import numpy as np



def build_feature_set(images: List[np.ndarray],
                      labels: List[int],
                      noise_type: str,
                      level: str,
                      variants: int = config.AUG_VARIANTS_PER_IMAGE,
                      seed: int = config.RANDOM_SEED
                      ) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    """
    Genera variantes aumentadas y extrae descriptores.

    Parámetros:
        images, labels : dataset base.
        noise_type     : 'gaussian' | 'salt_pepper'.
        level          : 'limpio' (sin ruido) | 'bajo' | 'medio' | 'alto'.
        variants       : nº de copias rotadas por imagen.

    Devuelve:
        X_by_desc : { 'hu': X_hu, 'zernike': X_z, 'shape_signature': X_ss }
                    cada X es (n_muestras, n_features).
        y         : etiquetas alineadas (n_muestras,).
        groups    : id de la imagen ORIGEN de cada muestra. Permite separar
                    train/test por imagen base (evita data leakage entre
                    variantes de la misma figura).
    """
    rng = np.random.default_rng(seed)
    feats: Dict[str, List[np.ndarray]] = {d: [] for d in DESCRIPTOR_NAMES}
    ys: List[int] = []
    groups: List[int] = []

    for gid, (img, lab) in enumerate(zip(images, labels)):
        for _ in range(variants):
            aug = random_rotation(img, rng)              # rotación 0..360
            if level != "limpio":
                aug = apply_noise(aug, noise_type, level, rng)  # ruido

            d = extract_all(aug)
            if d is None:
                continue   # figura no segmentable (ruido extremo): se descarta
            for name in DESCRIPTOR_NAMES:
                feats[name].append(d[name])
            ys.append(lab)
            groups.append(gid)

    X_by_desc = {name: np.asarray(v, dtype=np.float64) for name, v in feats.items()}
    return (X_by_desc,
            np.asarray(ys, dtype=np.int64),
            np.asarray(groups, dtype=np.int64))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC



def build_classifier(kind: str = "knn"):
    """
    Construye un pipeline (estandarización + clasificador).
        kind : 'knn' (K=7) | 'svm' (lineal).
    """
    if kind == "knn":
        clf = KNeighborsClassifier(n_neighbors=config.KNN_K)
    elif kind == "svm":
        clf = SVC(kernel="linear", C=1.0, random_state=config.RANDOM_SEED)
    else:
        raise ValueError(f"Clasificador desconocido: {kind}")

    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", clf),
    ])

## 5. Carga del dataset UPS-Writing-Skills (auto-detección en Kaggle)

Busca recursivamente en `/kaggle/input` (o en la carpeta local) las subcarpetas
`circle`, `square`, `triangle` que contengan imágenes. Así funciona sin importar
la estructura exacta del dataset montado.

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")

def load_ups_dataset(classes=None):
    """Carga UPS-Writing-Skills auto-detectando la ruta (Kaggle o local)."""
    classes = classes or config.UPS_CLASSES
    search_bases = [Path("/kaggle/input"), Path("data/ups_writing_skills"),
                    Path("../data/ups_writing_skills"), Path(".")]
    class_dir = {}
    for base in search_bases:
        if not base.exists():
            continue
        for d in base.rglob("*"):
            if not d.is_dir():
                continue
            name = d.name.lower()
            if name in classes and name not in class_dir:
                if any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file()):
                    class_dir[name] = d
        if all(c in class_dir for c in classes):
            break
    missing = [c for c in classes if c not in class_dir]
    if missing:
        raise FileNotFoundError(
            f"No se encontraron las carpetas {missing}. Añade el dataset "
            "'ups-writing-skills' con 'Add Data'. Rutas revisadas: "
            + ", ".join(str(b) for b in search_bases))

    images, labels = [], []
    for label, cls in enumerate(classes):
        files = sorted(f for f in class_dir[cls].iterdir()
                       if f.suffix.lower() in IMG_EXT)
        for f in files:
            img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
                labels.append(label)
    print("Rutas detectadas:")
    for c in classes:
        print(f"  {c:10s} -> {class_dir[c]}")
    return images, labels, list(classes)


images, labels, class_names = load_ups_dataset()
labels = np.array(labels)
print(f"\n{len(images)} imágenes | clases: {class_names}")

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for c, name in enumerate(class_names):
    i = int(np.where(labels == c)[0][0])
    axes[0, c].imshow(images[i], cmap="gray"); axes[0, c].set_title(name); axes[0, c].axis("off")
    axes[1, c].imshow(filled_shape_mask(images[i]), cmap="gray")
    axes[1, c].set_title(f"{name} (silueta)"); axes[1, c].axis("off")
plt.tight_layout(); plt.show()


## 6. Inyección de ruido (Gaussiano y Salt & Pepper, 3 niveles)
Se estudia cómo el ruido degrada las características globales (Unidad 3).

In [ ]:
rng = np.random.default_rng(0)
i = int(np.where(labels == 0)[0][0])   # un círculo
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, level in enumerate(["limpio", "bajo", "medio", "alto"]):
    g = images[i] if level == "limpio" else apply_noise(images[i], "gaussian", level, rng)
    sp = images[i] if level == "limpio" else apply_noise(images[i], "salt_pepper", level, rng)
    axes[0, col].imshow(g, cmap="gray"); axes[0, col].set_title(f"Gaussian {level}"); axes[0, col].axis("off")
    axes[1, col].imshow(sp, cmap="gray"); axes[1, col].set_title(f"S&P {level}"); axes[1, col].axis("off")
plt.tight_layout(); plt.show()


## 7. Rotaciones aleatorias (0º–360º) — verificación de invarianza

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for k, ang in enumerate([0, 45, 120, 210, 300]):
    axes[k].imshow(rotate(images[i], ang), cmap="gray")
    axes[k].set_title(f"{ang}°"); axes[k].axis("off")
plt.tight_layout(); plt.show()


## 8. Extracción de descriptores
Hu (7), Zernike n=8 (25 magnitudes) y Shape Signature (12 componentes).

In [ ]:
d = extract_all(images[i])
for k, v in d.items():
    print(f"{k:16s} -> dim={len(v)}")


## 9. Clasificación por nivel de ruido (KNN, K = 7)
Para cada nivel se construye el conjunto aumentado (rotaciones + ruido), se
extraen los tres descriptores y se entrena un KNN. El split es **por grupos**
(las variantes de una misma imagen no se reparten entre train/test) para evitar
*data leakage*. Se reportan **matrices de confusión** y **accuracy**.

> Cambia `NOISE` a `'salt_pepper'` para repetir el experimento con ruido impulsivo.

In [ ]:
LEVELS = ["limpio", "bajo", "medio", "alto"]
NOISE = "gaussian"    # cambia a "salt_pepper" para el otro experimento

robustness = {d: {} for d in DESCRIPTOR_NAMES}
for level in LEVELS:
    X_by, y, groups = build_feature_set(images, list(labels), NOISE, level)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=config.RANDOM_SEED)
    tr, te = next(gss.split(X_by["hu"], y, groups))
    for desc in DESCRIPTOR_NAMES:
        clf = build_classifier("knn"); clf.fit(X_by[desc][tr], y[tr])
        acc = accuracy_score(y[te], clf.predict(X_by[desc][te]))
        robustness[desc][level] = acc
    print(f"{level:7s} | " + " | ".join(f"{d}={robustness[d][level]:.3f}" for d in DESCRIPTOR_NAMES))


### Matriz de confusión (ejemplo: nivel ALTO, Zernike)

In [ ]:
X_by, y, groups = build_feature_set(images, list(labels), NOISE, "alto")
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=config.RANDOM_SEED)
tr, te = next(gss.split(X_by["zernike"], y, groups))
clf = build_classifier("knn"); clf.fit(X_by["zernike"][tr], y[tr])
cm = confusion_matrix(y[te], clf.predict(X_by["zernike"][te]))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues")
plt.title("Zernike — Gaussian alto"); plt.show()


## 10. Curva de robustez y PCA

In [ ]:
plt.figure(figsize=(6, 4))
for desc in DESCRIPTOR_NAMES:
    plt.plot(LEVELS, [robustness[desc][l] for l in LEVELS], marker="o", label=desc)
plt.ylim(0, 1.05); plt.grid(alpha=0.3); plt.xlabel("Nivel de ruido"); plt.ylabel("Accuracy")
plt.title(f"Robustez vs ruido {NOISE}"); plt.legend(); plt.show()

X_by, y, _ = build_feature_set(images, list(labels), NOISE, "limpio")
p = PCA(n_components=2).fit_transform(X_by["zernike"])
plt.figure(figsize=(5, 4))
for c, name in enumerate(class_names):
    m = y == c; plt.scatter(p[m, 0], p[m, 1], s=12, alpha=0.6, label=name)
plt.title("PCA — Zernike (limpio)"); plt.legend(); plt.show()


## 11. Conclusiones (alineadas con la guía)

- **Superioridad de la ortogonalidad:** los **Momentos de Zernike** mantienen una
  precisión alta y estable frente al ruido, mientras que los **Momentos de Hu**
  son los más sensibles (su accuracy es la más baja y la que más cae). Esto
  confirma que la base ortogonal de Zernike es menos redundante y más robusta.
- **Shape Signature** ofrece muy buena precisión porque trabaja sobre el contorno
  remuestreado y su firma espectral es naturalmente invariante a rotación/escala.
- **Degradación:** a mayor nivel de ruido, la calidad de la segmentación cae y con
  ella todos los descriptores; el filtro de mediana mitiga el ruido Salt & Pepper.
- El análisis **PCA** muestra cómo las tres clases forman clusters separables que el
  ruido tiende a mezclar.
